In [1]:
import torch, os
os.environ["CUDA_VISIBLE_DEVICES"] = "2" 

In [2]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. 設定檔案路徑 (請依據您的實際路徑修改)
# ==========================================
# 假設所有檔案都在同一個資料夾，且大檔是 .gz 格式
path_icustays = 'icustays.csv'
path_admissions = 'admissions.csv'
path_patients = 'patients.csv'
path_procedures = 'procedureevents.csv'       # 假設這個沒壓縮，若有請改 .csv.gz
path_text = 'processed_discharge.csv.gz'      # 您的文本檔
path_chartevents = 'chartevents.csv.gz'       # 您的生命徵象壓縮檔


In [3]:
# ==========================================
# 1. 建立核心骨幹 (Base Cohort) & Target
# ==========================================
print("=== 步驟 1: 建立核心骨幹 (急診 -> ICU) ===")

# 讀取 ICU 資料
df_icu = pd.read_csv(path_icustays, usecols=['subject_id', 'hadm_id', 'stay_id', 'intime', 'los'])

# 建立 Target: LOS > 7 天為 1，否則為 0
df_icu['target_long_stay'] = (df_icu['los'] > 7).astype(int)

# 讀取 Admission (用來篩選急診來源)
df_adm = pd.read_csv(path_admissions, usecols=['subject_id', 'hadm_id', 'admission_location'])

# 合併
df_cohort = pd.merge(df_icu, df_adm, on=['subject_id', 'hadm_id'], how='left')

# 篩選：只保留 'EMERGENCY ROOM'
df_cohort = df_cohort[df_cohort['admission_location'] == 'EMERGENCY ROOM'].copy()

# 清理不需要的欄位
df_cohort.drop(columns=['admission_location'], inplace=True)

print(f"步驟 1 完成。目前資料數據量 (Rows, Cols): {df_cohort.shape}")
print("-" * 30)


=== 步驟 1: 建立核心骨幹 (急診 -> ICU) ===
步驟 1 完成。目前資料數據量 (Rows, Cols): (37501, 6)
------------------------------


In [4]:
# ==========================================
# 2. 串接人口學資料 (Age, Gender)
# ==========================================
print("=== 步驟 2: 串接病患基本資料 ===")

df_pat = pd.read_csv(path_patients, usecols=['subject_id', 'gender', 'anchor_age', 'anchor_year'])

# 合併
df_cohort = pd.merge(df_cohort, df_pat, on='subject_id', how='left')

# 計算入 ICU 當下的年齡
# 公式: anchor_age + (ICU入帳年份 - anchor_year)
df_cohort['intime'] = pd.to_datetime(df_cohort['intime'])
df_cohort['admit_year'] = df_cohort['intime'].dt.year
df_cohort['age'] = df_cohort['anchor_age'] + (df_cohort['admit_year'] - df_cohort['anchor_year'])

# 性別編碼 (F=0, M=1) - 簡單處理
df_cohort['gender_code'] = df_cohort['gender'].apply(lambda x: 1 if x == 'M' else 0)

# 移除暫存欄位
df_cohort.drop(columns=['anchor_age', 'anchor_year', 'admit_year', 'gender'], inplace=True)

print(f"步驟 2 完成。目前資料數據量 (Rows, Cols): {df_cohort.shape}")
print("-" * 30)

=== 步驟 2: 串接病患基本資料 ===
步驟 2 完成。目前資料數據量 (Rows, Cols): (37501, 8)
------------------------------


In [10]:
import pandas as pd
import numpy as np

print("=== 步驟 3: 串接 Discharge Summary 文本 ===")

# ------------------------------------------
# 1. 讀取並清理文本資料
# ------------------------------------------
print("正在讀取文本資料...")
# 只讀取需要的欄位，節省記憶體
df_text = pd.read_csv(path_text, compression='gzip', usecols=['hadm_id', 'text'])

# 去除欄位名稱可能存在的空白 (例如 'text ' -> 'text')
df_text.columns = df_text.columns.str.strip()

# 確保是字串格式
df_text['text'] = df_text['text'].astype(str)

# 【關鍵】處理重複 Note：每個 hadm_id 只保留「字數最多」的那一篇
# 這樣我們確保了 df_text 裡的 hadm_id 是唯一的 (Unique Key)
df_text['text_len'] = df_text['text'].apply(len)
df_text_unique = df_text.sort_values('text_len', ascending=False).drop_duplicates(subset=['hadm_id'], keep='first')

# 移除暫存的長度欄位，只留我們要的兩欄
df_text_final = df_text_unique[['hadm_id', 'text']].copy()

# ------------------------------------------
# 2. 準備主表 (df_cohort)
# ------------------------------------------
# 確保 ID 型態一致 (這步不做很容易合併失敗)
df_cohort['hadm_id'] = df_cohort['hadm_id'].astype(int)
df_text_final['hadm_id'] = df_text_final['hadm_id'].astype(int)

# 【最關鍵的一步】合併前的大掃除
# 不管 df_cohort 以前有沒有跑過，強制移除所有可能衝突的文字欄位
cols_to_remove = ['text', 'text_x', 'text_y']
df_cohort = df_cohort.drop(columns=cols_to_remove, errors='ignore')

print("已清理主表中可能存在的舊文字欄位，準備合併...")

# ------------------------------------------
# 3. 執行合併 (Left Join)
# ------------------------------------------
# 經過上面的清理，這次合併保證只會有一個 'text' 欄位
df_cohort = pd.merge(df_cohort, df_text_final, on='hadm_id', how='left')

# ------------------------------------------
# 4. 後續處理
# ------------------------------------------
# 檢查合併是否成功
if 'text' in df_cohort.columns:
    print("合併成功！欄位 'text' 已建立。")
else:
    print("錯誤：合併後找不到 'text' 欄位，請檢查 hadm_id 是否正確。")

# 移除沒有 Note 的病人 (因為你的任務需要文字輸入)
initial_count = len(df_cohort)
df_cohort = df_cohort.dropna(subset=['text'])
final_count = len(df_cohort)

print("-" * 30)
print(f"處理結果摘要：")
print(f"原本人數: {initial_count}")
print(f"移除無 Note 後人數: {final_count} (移除了 {initial_count - final_count} 人)")
print(f"目前所有欄位: {df_cohort.columns.tolist()}")
print(f"目前的資料形狀 (Shape): {df_cohort.shape}")
print("-" * 30)

=== 步驟 3: 串接 Discharge Summary 文本 ===
正在讀取文本資料...
已清理主表中可能存在的舊文字欄位，準備合併...
合併成功！欄位 'text' 已建立。
------------------------------
處理結果摘要：
原本人數: 37501
移除無 Note 後人數: 35190 (移除了 2311 人)
目前所有欄位: ['subject_id', 'hadm_id', 'stay_id', 'intime', 'los', 'target_long_stay', 'age', 'gender_code', 'text']
目前的資料形狀 (Shape): (35190, 9)
------------------------------


In [13]:
# ==========================================
# 4. 串接插管資訊 (Intubation)
# ==========================================
print("=== 步驟 4: 串接插管處置 (Intubation) ===")

# 讀取處置檔 (假設是一般 CSV，若是壓縮檔請加 compression='gzip')
df_proc = pd.read_csv(path_procedures)

# 篩選插管 (Item ID: 224385) 且必須是我們 Cohort 裡的病人
target_stay_ids = df_cohort['stay_id'].unique()
df_intub = df_proc[
    (df_proc['itemid'] == 224385) & 
    (df_proc['stay_id'].isin(target_stay_ids))
].copy()

# 只要有紀錄就算有插管 (標記為 1)
intubated_stays = df_intub['stay_id'].unique()
df_cohort['is_intubated'] = df_cohort['stay_id'].isin(intubated_stays).astype(int)

print(f"步驟 4 完成。目前資料數據量 (Rows, Cols): {df_cohort.shape}")
print("-" * 30)


=== 步驟 4: 串接插管處置 (Intubation) ===
步驟 4 完成。目前資料數據量 (Rows, Cols): (35190, 10)
------------------------------


In [28]:
import os

# 設定合併後的輸出檔名
output_file = 'chartevents.csv.7z'

# 1. 找出所有分片檔案 (會抓取 .001, .002 ... 等結尾的檔案)
# 使用 sorted 確保順序正確 (001 -> 002 -> 003...)
parts = sorted([f for f in os.listdir('.') if f.startswith('chartevents.csv.7z.')])

if not parts:
    print("❌ 錯誤：找不到任何分片檔案，請確認檔案是否已上傳。")
else:
    print(f"✅ 找到 {len(parts)} 個分片檔案：")
    for p in parts: print(f"  - {p}")
    
    print(f"\n🚀 開始合併為 {output_file} ... (請稍候)")
    
    # 2. 執行二進位合併
    with open(output_file, 'wb') as outfile:
        for part in parts:
            print(f"   -> 正在寫入 {part} ...")
            with open(part, 'rb') as infile:
                # 分塊讀寫，避免記憶體爆掉
                while True:
                    chunk = infile.read(1024 * 1024 * 10) # 每次讀 10MB
                    if not chunk:
                        break
                    outfile.write(chunk)
    
    # 3. 檢查合併後的大小
    final_size_gb = os.path.getsize(output_file) / (1024**3)
    print("-" * 30)
    print(f"🎉 合併完成！")
    print(f"檔案名稱: {output_file}")
    print(f"檔案大小: {final_size_gb:.2f} GB")
    
    if final_size_gb > 3.0:
        print("✅ 大小正常，看起來合併成功了。")
    else:
        print("⚠️ 警告：合併後的檔案似乎有點小，請檢查分片是否完整。")

✅ 找到 7 個分片檔案：
  - chartevents.csv.7z.001
  - chartevents.csv.7z.002
  - chartevents.csv.7z.003
  - chartevents.csv.7z.004
  - chartevents.csv.7z.005
  - chartevents.csv.7z.006
  - chartevents.csv.7z.007

🚀 開始合併為 chartevents.csv.7z ... (請稍候)
   -> 正在寫入 chartevents.csv.7z.001 ...
   -> 正在寫入 chartevents.csv.7z.002 ...
   -> 正在寫入 chartevents.csv.7z.003 ...
   -> 正在寫入 chartevents.csv.7z.004 ...
   -> 正在寫入 chartevents.csv.7z.005 ...
   -> 正在寫入 chartevents.csv.7z.006 ...
   -> 正在寫入 chartevents.csv.7z.007 ...
------------------------------
🎉 合併完成！
檔案名稱: chartevents.csv.7z
檔案大小: 3.15 GB
✅ 大小正常，看起來合併成功了。


In [32]:
# ==========================================
# 5. 串接生命徵象 (7z 串流直讀模式 - 不解壓到硬碟)
# ==========================================
import pandas as pd
import numpy as np
import subprocess
import shutil

# 設定檔案名稱
path_7z = 'chartevents.csv.7z'

print("=== 步驟 5: 串接生命徵象 (雙重解壓修正版) ===")
print(f"正在建立資料水管，從 {path_7z} 讀取並同時解 GZIP...")

# 檢查 7z
if not shutil.which('7z'):
    raise RuntimeError("錯誤：找不到 '7z' 指令！請確認已安裝。")

target_items = [220181, 220210]
cohort_stay_ids = set(df_cohort['stay_id'].unique()) 
chunks_data = []
chunk_size = 1000000 

try:
    # 建立子程序
    command = ["7z", "e", "-so", path_7z]
    
    with subprocess.Popen(command, stdout=subprocess.PIPE) as proc:
        
        # 【修正點】加入 compression='gzip'
        # 因為 7z 吐出來的是 .gz 檔，所以我們要叫 Pandas 再解一次 gzip
        with pd.read_csv(proc.stdout, compression='gzip', chunksize=chunk_size, usecols=['stay_id', 'itemid', 'valuenum']) as reader:
            
            for i, chunk in enumerate(reader):
                filtered = chunk[
                    (chunk['stay_id'].isin(cohort_stay_ids)) & 
                    (chunk['itemid'].isin(target_items))
                ]
                
                if not filtered.empty:
                    chunks_data.append(filtered)
                
                if i % 10 == 0:
                    print(f"已處理 Chunk {i}...", end='\r')

    print("\n讀取完成！開始聚合計算...")

except Exception as e:
    print(f"\n[錯誤] {e}")
    print("如果還是失敗，請嘗試方案二 (直接讀取)。")

# --- 後續聚合計算 ---
if chunks_data:
    df_vitals_raw = pd.concat(chunks_data)
    
    # 計算 MAP Min
    map_min = df_vitals_raw[df_vitals_raw['itemid'] == 220181].groupby('stay_id')['valuenum'].min().rename('min_map')
    # 計算 RR Max
    rr_max = df_vitals_raw[df_vitals_raw['itemid'] == 220210].groupby('stay_id')['valuenum'].max().rename('max_resp_rate')
    
    # 合併回主表
    df_cohort = pd.merge(df_cohort, map_min, on='stay_id', how='left')
    df_cohort = pd.merge(df_cohort, rr_max, on='stay_id', how='left')
    print("生命徵象合併完成！")
else:
    print("警告：沒有讀到任何資料")

print(f"目前資料形狀: {df_cohort.shape}")

=== 步驟 5: 串接生命徵象 (雙重解壓修正版) ===
正在建立資料水管，從 chartevents.csv.7z 讀取並同時解 GZIP...
已處理 Chunk 430...
讀取完成！開始聚合計算...
生命徵象合併完成！
目前資料形狀: (35190, 14)


In [34]:
df_cohort.isnull().sum()

subject_id              0
hadm_id                 0
stay_id                 0
intime                  0
los                     0
target_long_stay        0
age                     0
gender_code             0
text                    0
is_intubated            0
min_map_x           33203
max_resp_rate_x     33192
min_map_y             292
max_resp_rate_y        28
dtype: int64

In [35]:
import pandas as pd

print("=== 最終資料清理與儲存 ===")
print(f"清理前資料維度: {df_cohort.shape}")

# 1. 刪除無效的 _x 欄位
# 這些是之前失敗的 merge 留下的空欄位
cols_to_drop = ['min_map_x', 'max_resp_rate_x']
df_cohort = df_cohort.drop(columns=cols_to_drop, errors='ignore')
print("已刪除 _x 欄位。")

# 2. 重新命名 _y 欄位
# 將有資料的 _y 欄位改回正確的名稱
rename_dict = {
    'min_map_y': 'min_map',
    'max_resp_rate_y': 'max_resp_rate'
}
df_cohort = df_cohort.rename(columns=rename_dict)
print("已重新命名 _y 欄位。")

# 3. 移除缺失值 (Drop NaNs)
# 根據您的圖片，大約會有 292 筆資料因為沒有 min_map 而被刪除
before_drop = len(df_cohort)
df_cohort = df_cohort.dropna()
after_drop = len(df_cohort)

print(f"已移除缺失值。")
print(f"刪除筆數: {before_drop - after_drop}")
print(f"剩餘筆數: {after_drop}")

# 4. 最終檢查
print("-" * 30)
print("最終欄位清單:")
print(df_cohort.columns.tolist())
print("\n缺失值檢查 (應該要全為 0):")
print(df_cohort.isnull().sum())

# 5. 儲存檔案
output_file = 'final_dataset.csv'
df_cohort.to_csv(output_file, index=False)
print("-" * 30)
print(f"✅ 恭喜！乾淨的訓練資料集已儲存為: {output_file}")
print("\n=== 資料預覽 ===")
cols_to_show = ['subject_id', 'target_long_stay', 'age', 'is_intubated', 'min_map', 'max_resp_rate']
print(df_cohort[cols_to_show].head())
print("\n缺失值狀況:")
print(df_cohort[cols_to_show].isnull().sum())

=== 最終資料清理與儲存 ===
清理前資料維度: (35190, 14)
已刪除 _x 欄位。
已重新命名 _y 欄位。
已移除缺失值。
刪除筆數: 308
剩餘筆數: 34882
------------------------------
最終欄位清單:
['subject_id', 'hadm_id', 'stay_id', 'intime', 'los', 'target_long_stay', 'age', 'gender_code', 'text', 'is_intubated', 'min_map', 'max_resp_rate']

缺失值檢查 (應該要全為 0):
subject_id          0
hadm_id             0
stay_id             0
intime              0
los                 0
target_long_stay    0
age                 0
gender_code         0
text                0
is_intubated        0
min_map             0
max_resp_rate       0
dtype: int64
------------------------------
✅ 恭喜！乾淨的訓練資料集已儲存為: final_dataset.csv

=== 資料預覽 ===
   subject_id  target_long_stay  age  is_intubated  min_map  max_resp_rate
0    10000032                 0   52             0     56.0           24.0
1    10000980                 0   76             0     83.0           25.0
2    10001217                 0   55             0     80.0           27.0
3    10001884                 1   77       

In [11]:
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# 1. 設定檔案路徑 (請依您的環境修改)
# ==========================================
path_final_data = 'final_dataset.csv'            # 您的分析檔案
path_admissions = 'admissions.csv'  # 原始 admissions 表格

print("--- 步驟 1: 讀取資料與補齊欄位 ---")

# 讀取 final_data
df_final = pd.read_csv(path_final_data)
print(f"已讀取 final_data.csv，資料筆數: {len(df_final)}")

# 讀取 admissions 的結局資訊
cols_to_add = ['subject_id', 'hadm_id', 'hospital_expire_flag', 'discharge_location']
print(f"正在從 {path_admissions} 讀取結局資訊...")
df_adm = pd.read_csv(path_admissions, usecols=cols_to_add)

# 合併
df_merged = pd.merge(df_final, df_adm, on=['subject_id', 'hadm_id'], how='left')

# 確保 target_long_stay 存在 (若無則自動計算)
if 'target_long_stay' not in df_merged.columns:
    if 'los' in df_merged.columns:
        df_merged['target_long_stay'] = (df_merged['los'] > 7).astype(int)
    else:
        print("錯誤：無法找到 target_long_stay 或 los 欄位！")

# ==========================================
# 2. 分析 Target 0 (短住院 <= 7天)
# ==========================================
print("\n=== [分析 A] Target 0 (短住院 <= 7天) ===")
df_target0 = df_merged[df_merged['target_long_stay'] == 0]
t0_total = len(df_target0)
t0_died = df_target0['hospital_expire_flag'].sum()
t0_survived = t0_total - t0_died
t0_death_rate = (t0_died / t0_total) * 100 if t0_total > 0 else 0

print(f"總人數: {t0_total}")
print(f"  - 存活出院: {t0_survived} ({100 - t0_death_rate:.2f}%)")
print(f"  - 院內死亡: {t0_died} ({t0_death_rate:.2f}%)")

# ==========================================
# 3. 分析 Target 1 (長住院 > 7天)
# ==========================================
print("\n=== [分析 B] Target 1 (長住院 > 7天) ===")
df_target1 = df_merged[df_merged['target_long_stay'] == 1]
t1_total = len(df_target1)
t1_died = df_target1['hospital_expire_flag'].sum()
t1_survived = t1_total - t1_died
t1_death_rate = (t1_died / t1_total) * 100 if t1_total > 0 else 0

print(f"總人數: {t1_total}")
print(f"  - 存活出院: {t1_survived} ({100 - t1_death_rate:.2f}%)")
print(f"  - 院內死亡: {t1_died} ({t1_death_rate:.2f}%)")

# ==========================================
# 4. 綜合比較表 (出院去向)
# ==========================================
print("\n=== [分析 C] 詳細出院去向比較 (前 5 名) ===")

# 建立交叉表
dispo_comparison = pd.crosstab(
    df_merged['target_long_stay'], 
    df_merged['discharge_location'], 
    normalize='index'
) * 100

# 整理表格
top_locations = df_merged['discharge_location'].value_counts().head(5).index
comparison_table = dispo_comparison[top_locations].T.round(2)
comparison_table.columns = ['Target 0 (短住院 %)', 'Target 1 (長住院 %)']

print(comparison_table)

# 簡單判斷
print("\n" + "="*30)
print("結果解讀建議：")
if t1_death_rate > t0_death_rate:
    print("1. 符合預期：長住院 (Target 1) 的死亡率較高。")
else:
    print("1. 異常注意：短住院 (Target 0) 的死亡率竟然比長住院還高？(請檢查是否包含早期死亡雜訊)")

print(f"2. Target 1 存活者中，轉往 SNF/Rehab 的比例通常較高 (目前顯示: {comparison_table.loc['SNF', 'Target 1 (長住院 %)'] if 'SNF' in comparison_table.index else 'N/A'}%)")

--- 步驟 1: 讀取資料與補齊欄位 ---
已讀取 final_data.csv，資料筆數: 34882
正在從 admissions.csv 讀取結局資訊...

=== [分析 A] Target 0 (短住院 <= 7天) ===
總人數: 31113
  - 存活出院: 27770 (89.26%)
  - 院內死亡: 3343 (10.74%)

=== [分析 B] Target 1 (長住院 > 7天) ===
總人數: 3769
  - 存活出院: 2939 (77.98%)
  - 院內死亡: 830 (22.02%)

=== [分析 C] 詳細出院去向比較 (前 5 名) ===
                          Target 0 (短住院 %)  Target 1 (長住院 %)
discharge_location                                          
HOME                                 28.60              6.72
HOME HEALTH CARE                     21.21              8.71
SKILLED NURSING FACILITY             19.12             13.06
DIED                                 10.77             22.01
REHAB                                 7.60             18.00

結果解讀建議：
1. 符合預期：長住院 (Target 1) 的死亡率較高。
2. Target 1 存活者中，轉往 SNF/Rehab 的比例通常較高 (目前顯示: N/A%)
